# Quantization Benchmarks: FP16 vs 8-bit vs 4-bit

This notebook demonstrates how to benchmark different quantization techniques and compare their:
- Inference speed
- Memory usage
- Cost savings

**Expected VRAM savings:**
- 8-bit: 30-50% reduction
- 4-bit: 60-75% reduction

## Setup

In [ ]:
import sys
sys.path.append('..')

from benchmark_quantization import run_quantization_benchmark
from visualize_results import create_all_visualizations
import json
import pandas as pd

## Run Benchmarks

We'll test a small model (OPT-350M) to demonstrate the concept. You can scale up to larger models.

In [ ]:
# Configuration
MODEL_NAME = "facebook/opt-350m"
PROMPT = "The future of artificial intelligence is"
MAX_TOKENS = 100
NUM_RUNS = 5
OUTPUT_FILE = "quantization_results.json"

In [ ]:
# Run the benchmark
print("Starting quantization benchmarks...")
print(f"Model: {MODEL_NAME}")
print(f"This may take several minutes...\n")

results = run_quantization_benchmark(
    model_name=MODEL_NAME,
    prompt=PROMPT,
    max_new_tokens=MAX_TOKENS,
    num_runs=NUM_RUNS,
    output_file=OUTPUT_FILE
)

## Analyze Results

In [ ]:
# Load results as DataFrame
with open(OUTPUT_FILE, 'r') as f:
    data = json.load(f)

df = pd.DataFrame(data)
df

In [ ]:
# Summary statistics
print("\n" + "="*60)
print("BENCHMARK SUMMARY")
print("="*60)

for _, row in df.iterrows():
    print(f"\n{row['quantization'].upper()}:")
    print(f"  Inference Time: {row['inference_time']:.2f}s")
    print(f"  Speedup: {row['speedup']:.2f}x")
    print(f"  VRAM: {row['vram_used_mb']:.0f} MB")
    print(f"  VRAM Savings: {row['vram_savings_percent']:.1f}%")
    print(f"  Tokens/sec: {row['tokens_per_second']:.1f}")

## Generate Visualizations

In [ ]:
# Create all visualizations
create_all_visualizations(
    results_file=OUTPUT_FILE,
    output_dir="visualizations",
    gpu_cost_per_hour=1.10  # A100 pricing
)

In [ ]:
# Display visualizations
from IPython.display import Image, display

print("\n📊 Quantization Comparison:")
display(Image('visualizations/quantization_comparison.png'))

print("\n💰 Cost Analysis:")
display(Image('visualizations/cost_analysis.png'))

print("\n📋 Summary Table:")
display(Image('visualizations/summary_table.png'))

## Cost Savings Analysis

Let's calculate potential cost savings at scale.

In [ ]:
# Cost calculation
GPU_COST_PER_HOUR = 1.10  # A100 pricing
INFERENCES_PER_MONTH = 100_000_000  # 100M inferences

print("\n" + "="*60)
print(f"MONTHLY COST PROJECTION ({INFERENCES_PER_MONTH:,} inferences)")
print("="*60)

baseline_time = df[df['quantization'] == 'fp16']['inference_time'].values[0]
baseline_cost = (baseline_time / 3600) * GPU_COST_PER_HOUR * INFERENCES_PER_MONTH

print(f"\nBaseline (FP16): ${baseline_cost:,.2f}")

for _, row in df.iterrows():
    if row['quantization'] != 'fp16':
        cost = (row['inference_time'] / 3600) * GPU_COST_PER_HOUR * INFERENCES_PER_MONTH
        savings = baseline_cost - cost
        savings_pct = (savings / baseline_cost) * 100
        
        print(f"\n{row['quantization'].upper()}:")
        print(f"  Monthly Cost: ${cost:,.2f}")
        print(f"  Savings: ${savings:,.2f} ({savings_pct:.1f}%)")

## Key Takeaways

### 8-bit Quantization
- ✅ Best balance of quality and efficiency
- ✅ ~40% VRAM reduction
- ✅ Minimal quality degradation
- ✅ Recommended for production

### 4-bit Quantization
- ✅ Maximum memory savings (~70%)
- ⚠️ Small quality trade-off
- ✅ Great for experimentation
- ✅ Enables larger models on smaller GPUs

### Cost Savings
- 💰 Quantization can save $20K-$40K monthly at scale
- 💰 Enables running larger models on the same hardware
- 💰 Reduces cloud GPU costs significantly

## Next Steps

1. Try with larger models (OPT-2.7B, Llama-2-7B)
2. Measure quality metrics (perplexity, accuracy)
3. Test with your specific use case
4. Combine with speculative decoding for more speedup